# VoxIntel — 02: Audio Format Validation

Goal: confirm what sample rate, channel count, and file format the SLURP
audio actually is, and whether it's directly compatible with
`facebook/wav2vec2-base-960h` (which expects 16kHz mono audio) — before we
write any ASR training code.

Assumes this notebook lives in `notebooks/`, run from the project root
(`VoxIntel/`), so that `from src.data import SLURPDataset` resolves.
If you're running the notebook from inside `notebooks/` itself, add the
project root to `sys.path` first (see Cell 0 below).


## Cell 0 — Make sure `src` is importable

In [12]:
import sys
from pathlib import Path

# If notebooks/ is not already run from the project root, add the root
# (one level up) to sys.path so "from src..." imports work.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


Project root: c:\Users\ACER\OneDrive\Desktop\VoxIntel


## Cell 1 — Load Dataset

In [13]:
from src.data import SLURPDataset

train = SLURPDataset(
    split="train",
    root_dir="C:\\Users\\ACER\\OneDrive\\Desktop\\VoxIntel\\data\\raw\\slurp"
)

len(train)


50628

## Cell 2 — Import Audio Utilities

In [14]:
from src.utils.audio import get_audio_stats

## Cell 3 — Inspect 20 Random Files

Expected output per file looks like:

```
sample_rate : 16000
channels    : 1
duration    : 2.3
format      : WAV
```

(SLURP audio is distributed as `.flac`, so don't be surprised if `format`
comes back as `FLAC` instead of `WAV` — that's expected and still fine for
Wav2Vec2, which just needs 16kHz mono, not a specific container format.)


In [15]:
import random

random.seed(42)
samples = random.sample(list(train), 20)

for sample in samples:
    stats = get_audio_stats(sample["audio_path"])

    print(sample["audio_path"])
    print(f"  sample_rate : {stats['sample_rate']}")
    print(f"  channels    : {stats['channels']}")
    print(f"  duration    : {stats['duration_sec']}")
    print(f"  format      : {stats['format']}")
    print("-" * 60)

C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\slurp\audio\slurp_real\audio-1490104957.flac
  sample_rate : 16000
  channels    : 1
  duration    : 2.815625
  format      : FLAC
------------------------------------------------------------
C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\slurp\audio\slurp_real\audio-1490704894.flac
  sample_rate : 16000
  channels    : 1
  duration    : 4.0954375
  format      : FLAC
------------------------------------------------------------
C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\slurp\audio\slurp_real\audio-1499267010-headset.flac
  sample_rate : 16000
  channels    : 1
  duration    : 2.304
  format      : FLAC
------------------------------------------------------------
C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\slurp\audio\slurp_real\audio-1490109155-headset.flac
  sample_rate : 16000
  channels    : 1
  duration    : 3.2
  format      : FLAC
------------------------------------------------------------
C:\Users\ACER\OneDrive\Desk

## Cell 4 — Check the Entire Dataset

Now check every file in the split. This reads only file headers (via
`soundfile.info` under the hood in `get_audio_stats`), so it's fast even
across tens of thousands of files — but on the full `train` split this can
still take a couple of minutes.

**What you're looking for:** ideally a single dominant value in each
counter — e.g. all files at `16000` Hz, all `1` channel. If so:

🎉 Wav2Vec2 will accept the dataset directly. No preprocessing needed.

If you see a mix of sample rates or channel counts, that's the signal you
need a resampling/mono-conversion step in your ASR data pipeline before
training (see `src/utils/audio.py`'s `load_audio(..., target_sr=16000)`,
which already handles this).


In [16]:
from collections import Counter
from tqdm import tqdm

sample_rates = Counter()
channels = Counter()
formats = Counter()
failed = []

for sample in tqdm(train, desc="Scanning audio files"):
    stats = get_audio_stats(sample["audio_path"])

    if stats["sample_rate"] is None:
        failed.append(sample["audio_path"])
        continue

    sample_rates[stats["sample_rate"]] += 1
    channels[stats["channels"]] += 1
    formats[stats["format"]] += 1

print("Sample Rates:", dict(sample_rates))
print("Channels    :", dict(channels))
print("Formats     :", dict(formats))
print(f"Unreadable files: {len(failed)}")
if failed:
    print("  e.g.:", failed[:5])


Scanning audio files: 100%|██████████| 50628/50628 [13:02<00:00, 64.71it/s]

Sample Rates: {16000: 50628}
Channels    : {1: 50628}
Formats     : {'FLAC': 50628}
Unreadable files: 0


## Cell 5 — Check for Weird Files

You already know from `reports/dataset_report.md` that some clips run up to
~23 seconds against a ~2.4s median. Here we pull out anything over 15
seconds so we can inspect them individually rather than just trusting the
aggregate stat.


In [17]:
long_audio = []

for sample in tqdm(train, desc="Finding long clips"):
    stats = get_audio_stats(sample["audio_path"])
    if stats["duration_sec"] is not None and stats["duration_sec"] > 15:
        long_audio.append({**sample, "duration_sec": stats["duration_sec"]})

print(f"Found {len(long_audio)} clips over 15 seconds")
len(long_audio)


Finding long clips: 100%|██████████| 50628/50628 [00:23<00:00, 2197.82it/s]

Found 9 clips over 15 seconds


9

In [18]:
# Inspect them: are these genuinely long utterances, or bad files?
for s in sorted(long_audio, key=lambda x: -x["duration_sec"])[:10]:
    print(f"{s['duration_sec']:.1f}s | intent={s['intent']} | {s['transcript']}")
    print(f"  {s['audio_path']}")


23.4s | intent=lists_query | what kind of lists do i have saved
  C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\slurp\audio\slurp_real\audio-1497621480-headset.flac
23.4s | intent=lists_query | what kind of lists do i have saved
  C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\slurp\audio\slurp_real\audio-1497621480.flac
19.9s | intent=general_quirky | find me a good wine shop that stock old wines like older than ten years navigate me to that shop
  C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\slurp\audio\slurp_real\audio-1497621444.flac
19.8s | intent=general_quirky | find me a good wine shop that stock old wines like older than ten years navigate me to that shop
  C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\slurp\audio\slurp_real\audio-1497621444-headset.flac
18.7s | intent=play_music | play for me music by madonna
  C:\Users\ACER\OneDrive\Desktop\VoxIntel\data\raw\slurp\audio\slurp_real\audio-1499696050.flac
18.0s | intent=social_post | i would want you to tweet for me

## Cell 6 — Listen to a Few Clips (Optional)

Run this in Jupyter (not as a plain script) to actually hear the audio.
Listen to ~10 clips and note anything that stands out:

- noisy background?
- accented speech?
- clipped / cut off audio?
- long silence at start/end?
- poor microphone quality?


In [19]:
from IPython.display import Audio, display

for i in range(10):
    sample = train[i]
    print(f"[{i}] intent={sample['intent']}  transcript: {sample['transcript']}")
    display(Audio(sample["audio_path"]))


[0] intent=calendar_set  transcript: event


[1] intent=calendar_set  transcript: event


[2] intent=calendar_set  transcript: event


[3] intent=calendar_set  transcript: event


[4] intent=calendar_set  transcript: event


[5] intent=calendar_set  transcript: event


[6] intent=calendar_set  transcript: event


[7] intent=calendar_set  transcript: event


[8] intent=calendar_set  transcript: event


[9] intent=calendar_set  transcript: i need an event three days from now scheduled with amy


### Listening notes

_Fill this in after listening:_

- Background noise: ...
- Accents: ...
- Clipping / cut-off audio: ...
- Silence padding: ...
- Microphone / recording quality: ...


## Cell 7 — Conclusion

_Update this paragraph with your actual findings from Cells 3–6 before
treating it as final._

All SLURP audio is sampled at 16 kHz, mono, and stored as compressed FLAC
files. This matches the expected input sample rate and channel count for
`facebook/wav2vec2-base-960h`, so no resampling or channel conversion is
required before baseline inference — only decoding the FLAC to a waveform
array, which `src/utils/audio.py`'s `load_audio()` already handles.

A small number of clips (see Cell 5) run well beyond the ~2.4s median
duration and may be worth excluding or capping during training for batching
efficiency, but they do not indicate a format problem.
